**Машинное обучение в экономике**

**Семинар 6. Эффекты воздействия**

Установка библиотек

In [1]:
# !python.exe -m pip install --upgrade pip
# !pip install numpy
# !pip install pandas
# !pip install scikit-learn
# !pip install openpyxl
# !pip install doubleml

In [2]:
# Подключим необходимые библиотеки
import numpy as np                                        # базовые операции с массивами
import pandas as pd                                       # базовые операции с датафреймами
import scipy as scipy
from copy import deepcopy
import math
from scipy.stats import multivariate_normal
import seaborn
import doubleml as dml
from sklearn.ensemble import RandomForestClassifier       # случайный лес (классификация)
from sklearn.ensemble import RandomForestRegressor        # случайный лес (регрессия)
from sklearn.ensemble import GradientBoostingClassifier   # градиентный бустинг (классификация)
from sklearn.ensemble import GradientBoostingRegressor    # градиентный бустинг (регрессия)
from sklearn.linear_model import Lasso                    # Лассо
from sklearn.base import clone
import statsmodels.api as sm                              # линейная регрессия
from sklearn.linear_model import LogisticRegression       # логистическая регрессия
from sklearn.model_selection import train_test_split      # разделение выборки

**Симуляция данных** 🐱

Цели ⭐

*   Симулировать контрольные переменные, переменную воздействия, инструментальную переменную и целевую переменную.

Симулируем данные по аналогии с предыдущим занятием.



Симулируем контрольные переменные $\text{Experience}_{i}$, $\text{Health}_{i}$ и $\text{Abilities}_{i}$ из нормального распределения с математическим ожиданием $\mu=25$ и стандартным отклонением $\sigma=10$, то есть $\text{N}\left(25, 10^2\right)$. Для удобства также усечем это распределение сверху и снизу числами $1$ и $50$ соответственно, а после этого округлим полученные значения.

In [3]:
# Число наблюдений
n = 10000

# Для воспроизводимости
np.random.seed(123)

# Опыт работы
experience = scipy.stats.norm.rvs(size = n,
                                  loc = 25, scale = 10)   # генерация
experience[experience >= 50] = 50                         # усечение
experience[experience <= 1]  = 1
experience = np.round(experience)                         # округление

# Здоровье
health = scipy.stats.norm.rvs(size = n,                   # генерация
                              loc = 25, scale = 10)
health[health >= 50] = 50                                 # усечение
health[health <= 1]  = 1
health               = np.round(health)                   # округление

# Способности
abilities = scipy.stats.norm.rvs(size = n,
                                 loc = 25, scale = 10)    # генерация
abilities[abilities >= 50] = 50                           # усечение
abilities[abilities <= 1]  = 1
abilities                  = np.round(abilities)          # округление

$$P(\text{Parents}_{i} = 1|\text{Health}_{i}, \text{Experience}_{i}) = F_{\text{Student}}\left(\ln\left(\text{Health}_{i}+\text{Experience}_{i}\right) - 4.3\right)$$

In [4]:
# Условная вероятность наличия у родителей высшего образования
parents_prob = scipy.stats.t.cdf(x = np.log(health + experience) - 4.3, df = 5)

# Факт наличия у родителей высшего образования
parents = np.random.binomial(n = 1, p = parents_prob, size = n)

Как и ранее рассмотрим следующую условную вероятность.

$$P(\text{Educ}_{i} = 1|\text{Experience}_{i}, \text{Health}_{i}, \text{Abilities}_{i}, \text{Parents}_{i}) = F_{\text{Logistic}}(2\times\sqrt{\text{Abilities}_{i}+\text{Experience}_{i}+\text{Health}_{i} + 20\times\text{Parents}_{i}} - 19)$$

Для того, чтобы впоследствии анализировать локальные средние эффекты воздействия $\text{LATE}$, необходимо различать величину переменной воздействия $\text{Educ}_{i}$ в зависимости от значения инструмента $\text{Parents}_{i}$. Для этого рассмотрим ни от чего не зависящую равномерную случайную величину $U_{i}\sim U(0,1)$ и введем гипотетические переменные:

$$\text{Educ}_{1i} = I(P(\text{Educ}_{i} = 1|\text{Experience}_{i}, \text{Health}_{i}, \text{Abilities}_{i}, \text{Parents}_{i} = 1)\geq U_{i})$$

$$\text{Educ}_{0i} = I(P(\text{Educ}_{i} = 1|\text{Experience}_{i}, \text{Health}_{i}, \text{Abilities}_{i}, \text{Parents}_{i} = 0)\geq U_{i})$$

Где:

$$I(\text{условие}) = \begin{cases}1\text{, если условие выполнено}\\0\text{, в противном случае}\end{cases}$$

Наблюдаемый уровень образования можно выразить как:

$$\text{Educ}_{i} = \begin{cases}\text{Educ}_{1i}\text{, если }\text{Parents}_{i} = 1\\ \text{Educ}_{0i}\text{, если }\text{Parents}_{i} = 0\end{cases}$$

Напомним, что к соблюдателям относятся те, у кого $\text{Educ}_{1i} > \text{Educ}_{0i}$, то есть получают высшее образование при $\text{Parents}_{i}=1$ и не получают - при $\text{Parents}_{i} = 0$.

In [5]:
# Условная вероятность наличия у индивида высшего образования

# Пороги
u = scipy.stats.uniform.rvs(size = n)

# Симулируем уровень образования индивидов
# с родителями с высшим образованием
parents1 = 1
educ_prob1 = scipy.stats.logistic.cdf(
    2 * np.sqrt(abilities + experience + health + 20 * parents1) - 19)
educ1 = (educ_prob1 >= u).astype(int)

# Симулируем уровень образования индивидов
# с родителями без высшего образования
parents0 = 0
educ_prob0 = scipy.stats.logistic.cdf(
    2 * np.sqrt(abilities + experience + health + 20 * parents0) - 19)
educ0 = (educ_prob0 >= u).astype(int)

# Соблюдатели
compliers = educ1 > educ0

# Факт наличия у индивида высшего образования
educ               = np.zeros(n)
educ[parents == 1] = educ1[parents == 1]
educ[parents == 0] = educ0[parents == 0]

# Доли людей с высшим образованием и соблюдателей
print(pd.DataFrame(data    = [np.mean(compliers), np.mean(educ)],
                   index   = ['P(compliers = 1)', 'P(educ = 1)'],
                   columns = ['Оценка']))

                  Оценка
P(compliers = 1)  0.3249
P(educ = 1)       0.3687


Уравнение зарплаты при отсутствии высшего образования.

$$\text{Wage}_{0i} = \underbrace{\text{Abilities}_{i} + 100 \times \frac{5\times\text{Experience}_{i} - 0.1\times \text{Experience}_{i}^2}{100 - \text{Health}_{i}}}_{g_{0}(X_{i})} + \varepsilon_{0i}$$

Уравнение зарплаты при наличии высшего образования.

$$\text{Wage}_{1i} = \underbrace{2 * \text{Abilities}_{i} + 3 \times\sqrt{\text{Experience}_{i}\times\text{Health}_{i}}}_{{g_{1}(X_{i})}} + \varepsilon_{1i}$$

Наблюдаемая заработная плата

$$\text{Wage}_{i} = \begin{cases}\text{Wage}_{1i}\text{, если }\text{Educ}_{i}=1\\ \text{Wage}_{0i}\text{, если }\text{Educ}_{i}=0\end{cases}$$

In [6]:
# Случайная ошибка
error_wage0 = scipy.stats.norm.rvs(size = n, scale = 10, loc = 0)
error_wage1 = scipy.stats.expon.rvs(size = n, scale = 10, loc = 0) - 10

# Функция от контрольных переменных
g0 = abilities + 100 * (5 * experience - 0.1 * experience ** 2) / (100 - health)
g1 = 2 * abilities + 3 * np.sqrt(experience * health)

# Зарплата в зависимости от наличия высшего образования
wage0 = g0 + error_wage0
wage1 = g1 + error_wage1

# Наблюдаемая зарплата
wage            = np.zeros(n)
wage[educ == 0] = wage0[educ == 0]
wage[educ == 1] = wage1[educ == 1]

Аггрегируем результаты симуляций в данные. При этом не будем включать в них $\text{Wage}_{1i}$ и $\text{Wage}_{0i}$, поскольку они не наблюдаются в реальной жизни одновременно для каждого индивида.

In [7]:
# Аггрегируем данные в датафрейм
df = pd.DataFrame({'educ': educ,       'experience': experience,
                   'health': health,   'abilities': abilities,
                   'parents': parents, 'wage': wage})

# Посмотрим на симулированные данные
df.head(10).style.format(precision = 2)

,educ,experience,health,abilities,parents,wage
0,0.00,14.00,13.00,35.00,0,97.61
1,1.00,35.00,22.00,38.00,0,162.49
2,0.00,28.00,17.00,26.00,0,97.34
3,1.00,10.00,49.00,34.00,0,130.18
4,0.00,19.00,32.00,5.00,0,91.84
5,1.00,42.00,27.00,18.00,1,159.11
6,0.00,1.00,20.00,40.00,0,56.35
7,0.00,21.00,14.00,23.00,1,94.24
8,1.00,38.00,25.00,29.00,0,143.04
9,0.00,16.00,21.00,27.00,0,114.36


**Оценивание эффектов воздействия с помощью гипотетических данных** 🐱

Цели ⭐

*   С помощью потенциальных исходов посчитать эффекты воздействия, а также оценить средний эффект воздействия, локальный средний эффект воздействия и условные средние эффекты воздействия.

Эффект воздействия:

$$\text{TE}_{i} = \text{Wage}_{1i} - \text{Wage}_{0i}$$

In [8]:
# Настоящие эффекты воздействия (не наблюдаются в данных)
TE = wage1 - wage0
print(TE[0:10])

[ 6.34704141 39.37664941 12.1504338  15.22401482 11.38033317 83.78031691
 28.9020707   8.08180497 54.08978838 -6.35566919]


Средний эффект воздействия:

$$\text{ATE} = \text{E}\left(\text{Wage}_{1i} - \text{Wage}_{0i}\right)$$

Если бы у нас были данные о $\text{Wage}_{1i}$ и $\text{Wage}_{0i}$, то мы могли бы очень точно оценить $\text{ATE}$ как:

$$\widehat{\text{ATE}} = \frac{1}{n}\sum\limits_{i=1}^{n}\text{Wage}_{1i} - \text{Wage}_{0i}$$

In [9]:
# Точное приближение среднего эффекта воздействия, то есть
# с помощью оценки, недоступной с помощью реальных данных
ATE = np.mean(TE)
print(ATE)

25.04627331263455


Локальный средний эффект воздействия:

$$\text{LATE} = \text{E}(\text{Wage}_{1i} - \text{Wage}_{0i} | \text{Educ}_{1i} > \text{Educ}_{0i})$$

In [10]:
# Точное приближение локального среднего эффекта воздействия, то есть
# с помощью оценки, недоступной с помощью реальных данных
LATE = np.mean(TE[compliers])
print(LATE)

25.724612885589327


Условный средний эффект воздействия:

$$\text{CATE}_{i} = \text{E}\left(\text{Wage}_{1i}|X_{i}\right) - \text{E}\left(\text{Wage}_{0i}|X_{i}\right) = g_{1}(X_{i}) - g_{0}(X_{i})$$

In [11]:
# Значения локальных средних эффектов воздействия
CATE = g1 - g0
print(CATE[0:10])

[17.54117821 53.93892925 17.23540522 21.97645831 -7.64467902 72.99735218
 47.29140786  3.62533111 60.66621004 13.13014885]


Средний эффект воздействия на подвергшихся воздействию:
$$\text{ATET} = \text{E}\left(\text{Wage}_{1i}|\text{Educ}_{i} = 1\right) - \text{E}\left(\text{Wage}_{0i}|\text{Educ}_{i} = 1\right)$$

In [12]:
# Значение среднего эффекта воздействия для подвергшихся воздействию
ATET = np.mean(TE[educ == 1])
print(ATET)

38.279291536941244


**Оценивание ATE как разницы в средних** 🐱

Цели ⭐

*   Оценить средний эффект воздействия опираясь на предпосылку о независимости.

Допущение о независимости:

$$\text{E}(\text{Wage}_{1i}|\text{Educ}_{i}=1) = \text{E}(\text{Wage}_{1i})\qquad \text{E}(\text{Wage}_{0i}|\text{Educ}_{i}=0) = \text{E}(\text{Wage}_{0i})$$

Попробуем оценить ATE наивным способом, опирающимся на допущение о независимости, которое не соблюдается в данном случае, поскольку $\text{Experience}_{i}$, $\text{Health}_{i}$ и $\text{Abilities}_{i}$ одновременно связаны и с наличием высшего образования $\text{Educ}_{i}$, и с заработной платой $\text{Wage}_{i}$.

Наивный подход предполагает оценивание $\text{ATE}$ как средней разницы в зарплатах людей с высшим образованием и без высшего образования.

$$\widehat{\text{ATE}}_{\text{naive}} = \frac{1}{n_{1}}\sum\limits_{i:\text{Educ}_{i}=1}\text{Wage}_{1i} - \frac{1}{n_{0}}\sum\limits_{i:\text{Educ}_{i}=0}\text{Wage}_{0i}$$

Где $n_{1}$ и $n_{0}$ - число людей с высшим образованием и без высшего образования соответственно.

In [13]:
# Наивная оценка как разница в выборочных средних
ATE_naive = np.mean(wage[educ == 1]) - np.mean(wage[educ == 0])

# Сравнение точного приближения и наивной оценки
print(pd.DataFrame(data    = [ATE, ATE_naive],
                   index   = ['ATE', 'ATE naive'],
                   columns = ['Оценка']))

              Оценка
ATE        25.046273
ATE naive  49.197187


**Оценивание ATE и CATE с помощью МНК** 🐱

Цели ⭐

*   С помощью МНК оценить ATE и CATE.
*   Сопоставить МНК оценку ATE с полученными ранее оценками.

Ослабим допущение о независимости до допущения об условной независимости:

$$\text{E}(\text{Wage}_{1i}|\text{Educ}_{i}=1,X_{i}) = \text{E}(\text{Wage}_{1i}|X_{i})\qquad \text{E}(\text{Wage}_{0i}|\text{Educ}_{i}=0,X_{i}) = \text{E}(\text{Wage}_{0i}|X_{i})$$

Попробуем оценить $\text{ATE}$ рассмотрев среднюю разницу в оценках зарплат, полученных с помощью МНК отдельно сперва по индивидам с образованием, а затем по индивидам без образования.

$$\widehat{\text{ATE}}_{\text{LS}} = \frac{1}{n}\sum\limits_{i=1}^{n} \underbrace{\hat{\text{E}}\left(\text{Wage}_{1i}|X_{i}\right) - \hat{\text{E}}\left(\text{Wage}_{0i}|X_{i}\right)}_{\widehat{\text{CATE}}_{i}}$$

Где:


*   $\hat{\text{E}}\left(\text{Wage}_{1i}|X_{i}\right)$ - оценка, полученная с использованием МНК оценок регрессионных коэффициентов $\beta$, полученных по выборке из индивидов с высшим образованием $\text{Educ}_{i} = 1$.
*   $\hat{\text{E}}\left(\text{Wage}_{0i}|X_{i}\right)$ - оценка, полученная с использованием МНК оценок регрессионных коэффициентов $\beta$, полученных по выборке из индивидов без высшего образования $\text{Educ}_{i} = 0$.



In [14]:
# Оценим средний эффект воздействия с помощью МНК

# МНК оценивание уравнения зарплаты для
# индивидов без высшего образования
y0  = df.loc[educ == 0, ['wage']]
x0  = df.loc[educ == 0, df.columns.drop(['wage', 'parents', 'educ'])]
x0  = sm.add_constant(x0)
ls0 = sm.OLS(y0, x0).fit()

# МНК оценивание уравнения зарплаты для
# индивидов с высшим образованием
y1  = df.loc[educ == 1, ['wage']]
x1  = df.loc[educ == 1, df.columns.drop(['wage', 'parents', 'educ'])]
x1  = sm.add_constant(x1)
ls1 = sm.OLS(y1, x1).fit()

# Оценим зарплаты при наличии и отсутствия высшего образования
# с помощью полученных МНК оценок
x = df.loc[:, df.columns.drop(['wage', 'parents', 'educ'])]
x = sm.add_constant(x)
  # МНК оценка E(wage0 | X) для всех индивидов
wage0_ls = ls0.predict(x)
  # МНК оценка E(wage1 | X) для всех индивидов
wage1_ls = ls1.predict(x)

# Оценки CATE
CATE_ls = np.array(wage1_ls - wage0_ls)

# Оценка ATE как средняя разница в прогнозах МНК оценок
ATE_ls = np.mean(CATE_ls)

In [15]:
# Сравним результаты
print(pd.DataFrame(data    = [ATE, ATE_naive, ATE_ls],
                   index   = ['ATE', 'ATE naive', "ATE ls"],
                   columns = ['Оценка']))

              Оценка
ATE        25.046273
ATE naive  49.197187
ATE ls     23.442768


**Оценивание ATE с помощью ДМО** 🐱

Цели ⭐

*   С помощью двойного машинного обучения (ДМО) оценить ATE.

Технический комментарий ⚡

Функция `DoubleMLIRM()` позволяет подготовиться к оцениваю средних эффектов воздействия $\text{ATE}$ с помощью ДМО при допущении об условной независимости (отсутствует эндогенность).

Основные аргументы:


*   `obj_dml_data` - данные, предварительно созданные с помощью функции `DoubleMLData()`.
*  ` ml_g` - метод машинного обучения, используемый для оценивания $\text{E}(Y_{i}|X_{i}, T_{i})$.
*  ` ml_m` - метод машинного обучения, используемый для оценивания $\text{E}(T_{i}|X_{i})$.
*  `n_folds` - на сколько частей разбивается выборка при кросс-фиттинге. Чем больше, тем, обычно, точнее, но дольше оценивание.
*  `rep` - сколько раз повторить оценивание.

Дополнительная информация может быть найдена в [документации](https://docs.doubleml.org/stable/api/generated/doubleml.DoubleMLIRM.html).

In [16]:
# Данные в формате, необходимом для применения DML
dml_standard_data = dml.DoubleMLData(
                            data   = df,
                            y_col  = 'wage',
                            d_cols = 'educ',
                            x_cols = ['experience', 'health', 'abilities'])

# Метод оценивания E(Y | X, T)
g_Y = RandomForestRegressor(n_estimators = 100,
                            max_depth    = 20,
                            max_features = 3)

# Метод оценивания E(T | X)
g_T = RandomForestClassifier(n_estimators = 100,
                             max_depth    = 20,
                             max_features = 3)

# Подготовка объекта
dml_standard = dml.DoubleMLIRM(obj_dml_data = dml_standard_data,
                               ml_g         = g_Y,
                               ml_m         = g_T,
                               n_rep        = 1,
                               n_folds      = 5)

# Оценим параметры
dml_standard.fit()

# Сохраним оценку
ATE_dml_standard = dml_standard.coef[0]

/Users/maria/Desktop/Code/.venv/lib/python3.12/site-packages/doubleml/utils/propensity_score_processing.py:272: UserWarning: Propensity predictions  from learner ml_m are close to zero or one (eps=1e-12).
  warnings.warn(
/Users/maria/Desktop/Code/.venv/lib/python3.12/site-packages/doubleml/double_ml.py:1636: UserWarning: The estimated nu2 for educ is not positive. Re-estimation based on riesz representer (non-orthogonal).
  warnings.warn(msg, UserWarning)


In [17]:
# Посмотрим на результат
print(dml_standard)

================== DoubleMLIRM Object ==================

------------------ Data Summary      ------------------
Outcome variable: wage
Treatment variable(s): ['educ']
Covariates: ['experience', 'health', 'abilities']
Instrument variable(s): None
No. Observations: 10000

------------------ Score & Algorithm ------------------
Score function: ATE

------------------ Machine Learner   ------------------
Learner ml_g: RandomForestRegressor(max_depth=20, max_features=3)
Learner ml_m: RandomForestClassifier(max_depth=20, max_features=3)
Out-of-sample Performance:
Regression:
Learner ml_g0 RMSE: [[11.26390066]]
Learner ml_g1 RMSE: [[11.14982772]]
Classification:
Learner ml_m Log Loss: [[0.61179206]]

------------------ Resampling        ------------------
No. folds: 5
No. repeated sample splits: 1

------------------ Fit Summary       ------------------
           coef   std err          t         P>|t|      2.5 %    97.5 %
educ  22.900126  1.487678  15.393204  1.818022e-53  19.984331  25.8

In [18]:
# Сопоставим результаты
print(pd.DataFrame(data    = [ATE, ATE_naive, ATE_ls, ATE_dml_standard],
                   index   = ['ATE', 'ATE naive', 'ATE ls', 'ATE dml standard'],
                   columns = ['Оценка']))

                     Оценка
ATE               25.046273
ATE naive         49.197187
ATE ls            23.442768
ATE dml standard  22.900126


**Важно** - в данном случае и далее точность оценок ATE и CATE можно заметно повысить с использованием тюнинга гиперпараметров. Однако, в данном случае соответствующая процедура опускается для краткости.

**Оценивание ATE и CATE с помощью T-learner** 🐱

Цели ⭐

*   С помощью T-learner оценить ATE и CATE.
*   Сопоставить T-learner оценку ATE с полученными ранее оценками.

Применим подход $\text{T-learner}$. При этом оценим не только $\text{ATE}$, но и условные эффекты воздействия $\text{CATE}_{i}$, определяемые как:

$$\text{CATE}_{i}=\text{E}\left(\text{Wage}_{1i}|X_{i}\right) - \text{E}\left(\text{Wage}_{0i}|X_{i}\right)$$

При этом средний эффект воздействия можно оценить как среднее оценок условных эффектов воздействия:

$$\widehat{\text{ATE}} = \frac{1}{n}\sum\limits_{i=1}^{n}\widehat{\text{CATE}}_{i}$$

В случае с T-learner эту оценку можно записать как:

$$\widehat{\text{ATE}}^{\text{T-learner}} = \frac{1}{n}\sum\limits_{i=1}^{n} \hat{\text{E}}\left(\text{Wage}_{i}|X_{i},\text{Educ}_{i}=1\right) - \hat{\text{E}}\left(\text{Wage}_{i}|X_{i},\text{Educ}_{i}=0\right)$$

Где:

*   $\hat{\text{E}}\left(\text{Wage}_{i}|X_{i},\text{Educ}_{i}=1\right)$ - оценка, полученная с использованием метода машинного обучения по выборке из индивидов с высшим образованием $\text{Educ}_{i} = 1$.
*   $\hat{\text{E}}\left(\text{Wage}_{i}|X_{i},\text{Educ}_{i}=0\right)$ - оценка, полученная с использованием метода машинного обучения по выборке из индивидов без высшего образования $\text{Educ}_{i} = 0$.

**Важно** - можно использовать разные методы для оценивания условных математчиеских ожиданий, например, оценить $\hat{\text{E}}\left(\text{Wage}_{i}|X_{i},\text{Educ}_{i}=1\right)$ с помощью градиентного бустинга, а $\hat{\text{E}}\left(\text{Wage}_{i}|X_{i},\text{Educ}_{i}=0\right)$ - случайным лесом.

In [19]:
# Оценивание ATE и CATE с помощью T-learner

# Подготовка модели: может быть и разной для тех кто с
# образованием и без образования
rf = RandomForestRegressor(n_estimators = 100,
                           max_depth    = 20,
                           max_features = 3)

# Обучение оценивать E(wage | X, educ = 0)
rf.fit(x0, y0)

# Оценки E(wage | X, educ = 0) для всех индивидов,
# в том числе для тех, у кого (educ = 1).
wage0_rf = rf.predict(x)

# Обучение оценивать E(wage | X, educ = 1)
rf.fit(x1, y1)

# Оценки E(wage | X, educ = 1) для всех индивидов,
# в том числе для тех, у кого (educ = 0).
wage1_rf = rf.predict(x)

# Оценки CATE
CATE_T = wage1_rf - wage0_rf

# Оценка ATE
ATE_T = np.mean(CATE_T)

/Users/maria/Desktop/Code/.venv/lib/python3.12/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/maria/Desktop/Code/.venv/lib/python3.12/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [20]:
# Сопоставим результаты
print(pd.DataFrame(data    = [ATE, ATE_naive, ATE_ls, ATE_dml_standard, ATE_T],
                   index   = ['ATE', 'ATE naive', 'ATE ls',
                              'ATE dml standard', 'ATE T-learner'],
                   columns = ['Оценка']))

                     Оценка
ATE               25.046273
ATE naive         49.197187
ATE ls            23.442768
ATE dml standard  22.900126
ATE T-learner     27.045980


**Оценивание ATE с помощью S-learner** 🐱

Цели ⭐

*   С помощью S-learner оценить ATE и CATE.
*   Сопоставить S-learner оценку ATE с полученными ранее оценками.

Оценка $\text{S-learner}$ метода будет иметь такой же вид, как и у $\text{T-learer}$:

$$\widehat{\text{CATE}}^{\text{S-learner}}_{i} = \frac{1}{n}\sum\limits_{i=1}^{n} \hat{\text{E}}\left(\text{Wage}_{i}|X_{i},\text{Educ}_{i}=1\right) - \hat{\text{E}}\left(\text{Wage}_{i}|X_{i},\text{Educ}_{i}=0\right)$$

Однако, методы оценивания самих условных математических ожиданий различаются:

*   $\hat{\text{E}}\left(\text{Wage}_{i}|X_{i},\text{Educ}_{i}=1\right)$ - оценка, полученная с использованием метода машинного обучения по всей выборке.
*   $\hat{\text{E}}\left(\text{Wage}_{i}|X_{i},\text{Educ}_{i}=0\right)$ - оценка, полученная с использованием метода машинного обучения по всей выборке.

In [21]:
# Оценивание ATE и CATE с помощью S-learner

# Подготовим данные
y = df.loc[:, ['wage']]
x = df.loc[:, df.columns.drop(['wage', 'parents'])]

# Подготовка модели: единая для тех у кого есть
# образование и для тех, у кого его нет
rf2 = RandomForestRegressor(n_estimators = 100,
                            max_depth    = 20,
                            max_features = 3)
rf2.fit(x, y)

# Оценки E(wage | X, educ = 0) для всех индивидов,
# в том числе для тех, у кого (educ = 1).
x0         = deepcopy(x)
x0["educ"] = 0
wage0_rf2  = rf2.predict(x0)

# Оценки E(wage | X, educ = 1) для всех индивидов,
# в том числе для тех, у кого (educ = 0).
x1         = deepcopy(x)
x1["educ"] = 1
wage1_rf2  = rf2.predict(x1)

# Оценки CATE
CATE_S = wage1_rf2 - wage0_rf2

# Оценка ATE
ATE_S = np.mean(CATE_S)

/Users/maria/Desktop/Code/.venv/lib/python3.12/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [22]:
# Сопоставим результаты
print(pd.DataFrame(data    = [ATE, ATE_naive, ATE_ls, ATE_dml_standard,
                              ATE_T, ATE_S],
                   index   = ['ATE', 'ATE naive', 'ATE ls', 'ATE dml standard',
                              'ATE T-learner', 'ATE S-learner'],
                   columns = ['Оценка']))

                     Оценка
ATE               25.046273
ATE naive         49.197187
ATE ls            23.442768
ATE dml standard  22.900126
ATE T-learner     27.045980
ATE S-learner     26.935177


**Оценивание ATE с помощью взвешивания на обратные вероятности** 🐱

Цели ⭐

*   С помощью взвешивания на обратные вероятности (IPW - inverse probability weighting) оценить ATE и CATE.
*   Сопоставить IPW оценку ATE с полученными ранее оценками.

Оценка, получаемая с помощью взвешивания на обратные вероятности $\text{IPW}$, имеет вид:

$$\widehat{\text{ATE}}^{\text{IPW}} = \frac{1}{n}\sum\limits_{i=1}^{n}\frac{\text{Educ}_{i}\times\text{Wage}_{i}}{\hat{P}\left(\text{Educ}_{i}=1|X_{i}\right)} - \frac{\left(1 - \text{Educ}_{i}\right)\times\text{Wage}_{i}}{1 - \hat{P}\left(\text{Educ}_{i}=1|X_{i}\right)}$$

Где условные вероятности $\hat{P}\left(\text{Educ}_{i}=1|X_{i}\right)$ оценивается с помощью методов классификации, например, градиентного бустинга или логистической регрессии.

In [23]:
# Оценивание с помощью обратного взвешивания на вероятности IPW

# Подготовим данные
target   = df.loc[:, ['educ']]
features = df.loc[:, df.columns.drop(['wage', 'educ', 'parents'])]

# Подготовим метод машинного обучения
gb = GradientBoostingClassifier(loss          = 'log_loss',
                                n_estimators  = 100,
                                learning_rate = 0.1)
gb.fit(features, target)

# Оценим условные вероятности P(educ = 1 | X)
prob_gb = gb.predict_proba(features)[:, 1]

# Оценим псевдоисходы
wage_pseudo = (wage * educ) / prob_gb - (wage * (1 - educ)) / (1 - prob_gb)

# Оценим ATE
ATE_IPW = np.mean(wage_pseudo)

/Users/maria/Desktop/Code/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [24]:
# Сопоставим результаты
print(pd.DataFrame(data    = [ATE, ATE_naive, ATE_ls, ATE_dml_standard,
                              ATE_T, ATE_S, ATE_IPW],
                   index   = ['ATE', 'ATE naive', 'ATE ls', 'ATE dml standard',
                              'ATE T-learner', 'ATE S-learner', 'ATE IPW'],
                   columns = ['Оценка']))

                     Оценка
ATE               25.046273
ATE naive         49.197187
ATE ls            23.442768
ATE dml standard  22.900126
ATE T-learner     27.045980
ATE S-learner     26.935177
ATE IPW           21.492720


**Оценивание CATE с использованием метода трансформации классов** 🐱

Цели ⭐

*   С помощью метода трансформации класов (CT) оценить CATE.

Методом трансформации классов условные эффекты воздействия оцениваются как:
$$\widehat{\text{CATE}}_{i}^{\text{IPW}}=\hat{\text{E}}(\widehat{\text{Wage}}_{i}^{*}|X_{i})$$

Где $\hat{\text{E}}(\widehat{\text{Wage}}_{i}^{*}|X_{i})$ это оценка условного математического ожидания оценки псевдоисхода, определяемой как:

$$\widehat{\text{Wage}}_{i}^{*} = \frac{\text{Educ}_{i}\times\text{Wage}_{i}}{\hat{P}\left(\text{Educ}_{i}=1|X_{i}\right)} - \frac{\left(1 - \text{Educ}_{i}\right)\times\text{Wage}_{i}}{1 - \hat{P}\left(\text{Educ}_{i}=1|X_{i}\right)}$$

In [25]:
# Оценим CATE методом трансформации классов

# Метод прогнозирования условных математических ожиданий оценок псевдоисходов
rf3 = RandomForestRegressor(n_estimators = 100,
                            max_depth    = 5,
                            max_features = 3)
rf3.fit(features, wage_pseudo)

# Оценки CATE (метод трансформации классов)
CATE_CT = rf3.predict(features)

**Оценивание ATE с помощью метода с двойной устойчивостью** 🐱

Цели ⭐

*   С помощью метода с двойной устойчивостью (DR - doubly robust) оценить ATE и CATE.
*   Сопоставить DR оценку ATE с полученными ранее оценками.

Методом с двойной устойчивостью средний эффект воздействия оценивается как:

$$\widehat{\text{ATE}}^{\text{DR}} = \frac{1}{n}\sum\limits_{i=1}^{n}\hat{\text{E}}\left(\text{Wage}_{i}|X_{i}, T_{i}=1\right) - \hat{\text{E}}\left(\text{Wage}_{i}|X_{i}, T_{i}=0\right) + \frac{\text{Educ}_{i}\times\left(\text{Wage}_{i}-\hat{\text{E}}\left(\text{Wage}_{i}|X_{i}, T_{i}=1\right)\right)}{\hat{P}(\text{Educ}_{i}=1|X_{i})} - \frac{\left(1-\text{Educ}_{i}\right)\times\left(\text{Wage}_{i}-\hat{\text{E}}\left(\text{Wage}_{i}|X_{i}, T_{i}=0\right)\right)}{1 - \hat{P}(\text{Educ}_{i}=1|X_{i})}$$

In [26]:
# Оценим ATE с помощью DR метода с двойной устойчивостью
ATE_DR = np.mean((wage1_rf2 - wage0_rf2) + \
                 educ * (wage - wage1_rf2) / prob_gb - \
                 (1 - educ) * (wage - wage0_rf2) / (1 - prob_gb))

In [27]:
# Сопоставим результаты
print(pd.DataFrame(data    = [ATE, ATE_naive, ATE_ls, ATE_dml_standard,
                              ATE_T, ATE_S, ATE_IPW, ATE_DR],
                   index   = ['ATE', 'ATE naive', 'ATE ls', 'ATE dml standard',
                              'ATE T-learner', 'ATE S-learner',
                              'ATE IPW', 'ATE DR'],
                   columns = ['Оценка']))

                     Оценка
ATE               25.046273
ATE naive         49.197187
ATE ls            23.442768
ATE dml standard  22.900126
ATE T-learner     27.045980
ATE S-learner     26.935177
ATE IPW           21.492720
ATE DR            26.636510


**Сравнение точности оценок CATE** 🐱

In [28]:
# Объединим полученные ранее оценки CATE
CATE_mat = pd.DataFrame({'True': CATE, 'LS': CATE_ls, 'T-learner': CATE_T,
                         'S-learner': CATE_S, 'IPW': CATE_CT})
print(CATE_mat)

           True         LS  T-learner  S-learner        IPW
0     17.541178  14.853815  18.996654  21.723396  12.891152
1     53.938929  43.477719  49.492014  51.985491  60.401259
2     17.235405  21.796421  24.467643  24.076889  22.217621
3     21.976458  33.860006  31.171086  32.533227  67.832351
4     -7.644679   2.576233   7.904985   9.193257 -14.087543
...         ...        ...        ...        ...        ...
9995  -6.812260  -2.076747   3.284638   2.878097 -40.775009
9996  20.669048  12.069559  26.852784  30.285119  21.774749
9997  -8.918539  -1.484362   3.155328   5.216409 -59.964980
9998  28.202630  30.588160  21.106696  20.440147  22.101389
9999  63.633532  52.030354  59.730402  59.716135  86.184612

[10000 rows x 5 columns]


Сравним точность оценок по среднеквадратической ошибке (недоступно на реальных данных):

$$\text{MSE}_{0} = \frac{1}{n}\sum\limits_{i=1}^{n}\left(\text{CATE}_{i}-\widehat{\text{CATE}}_{i}\right)^2$$

In [29]:
# Сравнение оценок CATE на основании истинных значений
CATE_MSE0 = pd.DataFrame(data    = [np.mean((CATE_ls  - CATE) ** 2),
                                    np.mean((CATE_T   - CATE) ** 2),
                                    np.mean((CATE_S   - CATE) ** 2),
                                    np.mean((CATE_CT  - CATE) ** 2)],
                         index   = ['LS', 'T-learner', 'S-learner', 'CT'],
                         columns = ['MSE0'])
print(CATE_MSE0)

                 MSE0
LS         299.804552
T-learner  100.020710
S-learner   94.511639
CT         298.954580


Сравним точность оценок с помощью псевдоисходов:

$$\text{MSE}^{*} = \frac{1}{n}\sum\limits_{i=1}^{n}\left(\text{Wage}_{i}^{*}-\widehat{\text{CATE}}_{i}\right)^2$$

In [30]:
# Сравнение CATE на основании псевдоисходов
CATE_MSE1 = pd.DataFrame(data    = [np.mean((wage_pseudo - CATE_ls) ** 2),
                                    np.mean((wage_pseudo - CATE_T) ** 2),
                                    np.mean((wage_pseudo - CATE_S) ** 2),
                                    np.mean((wage_pseudo - CATE_CT) ** 2)],
                         index   = ['LS', 'T-learner', 'S-learner', 'CT'],
                         columns = ['MSE1'])
print(CATE_MSE1)

                   MSE1
LS         74992.388679
T-learner  74848.847823
S-learner  74837.005368
CT         73225.611825


**Проблема** - метод трансформации классов оценивает условные математические ожидания с использованием псевдоисходов, поэтому его точность на обучающей выборке может быть завешена.

**Решение** - провести сравнение на тестовой выборке.

Для кракости сравним лишь S-learner и метод трансформации классов.

In [31]:
# Разобьем выборку на обучающую и тестовую
features_train, features_test, educ_train, educ_test, wage_train, wage_test = train_test_split(
    features, df['educ'], df['wage'], test_size = 0.2, random_state = 777)

# Оценим условные вероятности P(educ = 1 | X)
gb.fit(features_train, educ_train)
prob_gb_train = gb.predict_proba(features_train)[:, 1]
prob_gb_test  = gb.predict_proba(features_test)[:, 1]

# Оценим псевдоисходы
wage_pseudo_train = (wage_train * educ_train) / prob_gb_train - \
                    (wage_train * (1 - educ_train)) / (1 - prob_gb_train)
wage_pseudo_test  = (wage_test * educ_test) / prob_gb_test - \
                    (wage_test * (1 - educ_test)) / (1 - prob_gb_test)

# Оценки CATE метод S-learner
rf2.fit(features_train, wage_pseudo_train)
CATE_S_test = rf2.predict(features_test)

# Оценки CATE методом трансформации классов
rf3.fit(features_train, wage_pseudo_train)
CATE_CT_test = rf3.predict(features_test)

# Сравнение CATE на основании псевдоисходов на тестовой выборке
CATE_MSE1_test = pd.DataFrame(data     = [np.mean((wage_pseudo_test - CATE_S_test) ** 2),
                                          np.mean((wage_pseudo_test - CATE_CT_test) ** 2)],
                              index    = ['S-learner', 'CT'],
                              columns  = ['MSE1'])
print(CATE_MSE1_test)

                   MSE1
S-learner  97718.022225
CT         86463.478042


**Оценивание LATE с помощью ДМО в условиях эндогенности** 🐱

Цели ⭐

*   Оценить LATE с помощью двойного машинного обучения без инструментальной переменной.
*   Оценить LATE с помощью двойного машинного обучения с инструментальной переменной.
*   Сравнить точность оценок.

Представим, что переменная $\text{Abilities}_{i}$ отсутствует в данных, из-за чего возникает эндогенность, поскольку эта переменная влияет и на зарплату $\text{Wage}_{i}$, и на вероятность получения высшего образования $\text{Educ}_{i}$.

Напомним, что при наличии эндогенности с помощью инструментальных переменных мы можем оценить лишь $\text{LATE}$, а не $\text{ATE}$.

Попробуем сперва воспользоваться ДМО методом без инструментальных переменных, чтобы продемонстрировать проблемы, возникающие при игнорировании эндогенности.

In [32]:
# Данные в формате, необходимом для применения DML
dml_standard2_data = dml.DoubleMLData(
                             data   = df,
                             y_col  = 'wage',
                             d_cols = 'educ',
                             x_cols = ['experience', 'health'])

# Подготовка объекта
dml_standard2 = dml.DoubleMLIRM(obj_dml_data = dml_standard2_data,
                                ml_g         = g_Y,
                                ml_m         = g_T,
                                n_rep        = 1,
                                n_folds      = 5)

# Оценим параметры
dml_standard2.fit()

# Посмотрим на результат
print(dml_standard2)

# Сохраним оценку
LATE_dml_standard2 = dml_standard2.coef[0]

================== DoubleMLIRM Object ==================

------------------ Data Summary      ------------------
Outcome variable: wage
Treatment variable(s): ['educ']
Covariates: ['experience', 'health']
Instrument variable(s): None
No. Observations: 10000

------------------ Score & Algorithm ------------------
Score function: ATE

------------------ Machine Learner   ------------------
Learner ml_g: RandomForestRegressor(max_depth=20, max_features=3)
Learner ml_m: RandomForestClassifier(max_depth=20, max_features=3)
Out-of-sample Performance:
Regression:
Learner ml_g0 RMSE: [[15.28310738]]
Learner ml_g1 RMSE: [[23.75662679]]
Classification:
Learner ml_m Log Loss: [[0.67404712]]

------------------ Resampling        ------------------
No. folds: 5
No. repeated sample splits: 1

------------------ Fit Summary       ------------------
           coef   std err          t         P>|t|      2.5 %     97.5 %
educ  38.720761  3.637818  10.643952  1.860668e-26  31.590768  45.850753


/Users/maria/Desktop/Code/.venv/lib/python3.12/site-packages/doubleml/utils/propensity_score_processing.py:272: UserWarning: Propensity predictions  from learner ml_m are close to zero or one (eps=1e-12).
  warnings.warn(
/Users/maria/Desktop/Code/.venv/lib/python3.12/site-packages/doubleml/double_ml.py:1636: UserWarning: The estimated nu2 for educ is not positive. Re-estimation based on riesz representer (non-orthogonal).
  warnings.warn(msg, UserWarning)


Теперь воспользуемся ДМО методом оценивания $\text{LATE}$ с помощью инструментальных переменных.

Техническое примечание ⚡

Чтобы добавить инструментальную переменную, необходимо указать ее название в качестве значения параметра `z_cols` функции `DoubleMLData()`.

Для оценивания $\text{LATE}$ с помощью ДМО с инструментальными переменными используется функция `DoubleMLIIVM()`, в которой метод оценивания $\text{E}(Z_{i} | X_{i})$ подается через аргумент `ml_m`. Подробная информация об этой функции может быть найдена в [документации](https://docs.doubleml.org/stable/api/generated/doubleml.DoubleMLIIVM.html).

In [33]:
# Данные в формате, необходимом для применения DML
dml_iv_data = dml.DoubleMLData(data   = df,
                               y_col  = 'wage',
                               d_cols = 'educ',
                               z_cols = 'parents',
                               x_cols = ['experience', 'health'])

# Метод оценивания E(Z | X)
g_Z = GradientBoostingClassifier(loss          = 'log_loss',
                                 n_estimators  = 100,
                                 learning_rate = 0.1)

# Подготовка объекта
dml_iv = dml.DoubleMLIIVM(obj_dml_data = dml_iv_data,
                          ml_g         = g_Y,
                          ml_m         = g_Z,
                          ml_r         = g_T,
                          n_rep        = 1,
                          n_folds      = 5)

# Оценим параметры
dml_iv.fit()

# Посмотрим на результат
print(dml_iv)

# Сохраним оценку
LATE_dml_iv = dml_iv.coef[0]

================== DoubleMLIIVM Object ==================

------------------ Data Summary      ------------------
Outcome variable: wage
Treatment variable(s): ['educ']
Covariates: ['experience', 'health']
Instrument variable(s): ['parents']
No. Observations: 10000

------------------ Score & Algorithm ------------------
Score function: LATE

------------------ Machine Learner   ------------------
Learner ml_g: RandomForestRegressor(max_depth=20, max_features=3)
Learner ml_m: GradientBoostingClassifier()
Learner ml_r: RandomForestClassifier(max_depth=20, max_features=3)
Out-of-sample Performance:
Regression:
Learner ml_g0 RMSE: [[27.15521782]]
Learner ml_g1 RMSE: [[29.23912246]]
Classification:
Learner ml_m Log Loss: [[0.63364096]]
Learner ml_r0 Log Loss: [[1.28834857]]
Learner ml_r1 Log Loss: [[1.15864269]]

------------------ Resampling        ------------------
No. folds: 5
No. repeated sample splits: 1

------------------ Fit Summary       ------------------
           coef   std 

In [34]:
# Сопоставим результаты
print(pd.DataFrame(data    = [ATE, LATE, LATE_dml_standard2, LATE_dml_iv],
                   index   = ['ATE', 'LATE', 'LATE dml standard2', 'LATE dml iv',],
                   columns = ['Оценка']))

                       Оценка
ATE                 25.046273
LATE                25.724613
LATE dml standard2  38.720761
LATE dml iv         25.838161


**Анализ реальных данных** 🐱

Подключим пакеты

In [35]:
# !pip install rpy2
# ! add-apt-repository -y ppa:cran/imagemagick
# ! apt-get update
# ! apt-get install -y libmagick++-dev
# ! R -e "install.packages('switchSelection')"

In [36]:
import rpy2                                               # R в python
import rpy2.robjects as ro
import rpy2.robjects.packages as rpackages
from rpy2.robjects.vectors import StrVector
from rpy2.robjects.packages import importr, data
from rpy2.robjects import pandas2ri
from rpy2.robjects import IntVector, Formula

ModuleNotFoundError: No module named 'rpy2'

Установим необходимые библиотеки из `R`

In [ ]:
# Пакет для установки пакетов из R
utils = rpackages.importr('utils')
utils.chooseCRANmirror(ind = 1)

# Пакеты с базовым функционалом R
base  = rpackages.importr('base')
stats = rpackages.importr('stats')

# Подключение пакета
switchSelection = importr('switchSelection')

# Активация конвертации
pandas2ri.activate()

# Во избежание проблем с версиями pandas
pd.DataFrame.iteritems = pd.DataFrame.items

Загрузим данные

In [ ]:
# Достанем данные из библиотеки
utils.data("cps", package = "switchSelection")
cps = base.get("cps")

# Посмотрим на данные
cps.head(10).style.format(precision = 2)

In [ ]:
# Информация о данных
print(utils.help('cps'))

In [ ]:
# Создадим переменную на высшее образование для индивида и супруга
cps['university']  = cps['bachelor'] + cps['master']
cps['suniversity'] = cps['sbachelor'] + cps['smaster']

In [ ]:
# Уберем пропуски по целевой переменной, переменной воздействия и
# инструментальной переменной
cps = cps[~(cps['lwage'].isna() | cps['university'].isna() |
            cps['suniversity'].isna())]

Оценим эффекты воздействия

In [ ]:
# Оценим средний эффект воздействия образования на заработную плату
# наивным способом
ATE_naive = np.mean(cps['lwage'][cps['university'] == 1]) - \
            np.mean(cps['lwage'][cps['university'] == 0])
print(ATE_naive)

In [ ]:
# Оценим средний эффект воздействия с помощью МНК

# Отберем лишь нужные переменные
df = cps[['lwage', 'university', 'age', 'health', 'nchild', 'suniversity']]

# МНК оценивание уравнения зарплаты для
# индивидов без высшего образования
y0  = df.loc[df['university'] == 0, ['lwage']]
x0  = df.loc[df['university'] == 0,
             df.columns.drop(['lwage', 'university', 'suniversity'])]
x0  = sm.add_constant(x0)
ls0 = sm.OLS(y0, x0).fit()

# МНК оценивание уравнения зарплаты для
# индивидов с высшим образованием
y1  = df.loc[df['university'] == 1, ['lwage']]
x1  = df.loc[df['university'] == 1,
             df.columns.drop(['lwage', 'university', 'suniversity'])]
x1  = sm.add_constant(x1)
ls1 = sm.OLS(y1, x1).fit()

# Оценим зарплаты при наличии и отсутствия высшего образования
# с помощью полученных МНК оценок
x = df.loc[:, df.columns.drop(['lwage', 'university', 'suniversity'])]
x = sm.add_constant(x)
  # МНК оценка E(wage0 | X) для всех индивидов
wage0_ls = ls0.predict(x)
  # МНК оценка E(wage1 | X) для всех индивидов
wage1_ls = ls1.predict(x)

# Оценки CATE
CATE_ls = np.array(wage1_ls - wage0_ls)

# Оценка ATE как средняя разница в прогнозах МНК оценок
ATE_ls = np.mean(CATE_ls)

# Результаты МНК оценивания
print(ls0.summary())
print(ls1.summary())

In [ ]:
# Данные в формате, необходимом для применения DML
dml_standard_data = dml.DoubleMLData(
                            data   = cps,
                            y_col  = 'lwage',
                            d_cols = 'university',
                            x_cols = ['age', 'health', 'nchild'])

# Метод оценивания E(Y | X, T)
g_Y = RandomForestRegressor(n_estimators = 100,
                            max_depth    = 10,
                            max_features = 2)

# Метод оценивания E(T | X)
g_T = RandomForestClassifier(n_estimators = 100,
                             max_depth    = 10,
                             max_features = 2)

# Подготовка объекта
dml_standard = dml.DoubleMLIRM(obj_dml_data = dml_standard_data,
                               ml_g         = g_Y,
                               ml_m         = g_T,
                               n_rep        = 1,
                               n_folds      = 5)

# Оценим параметры
dml_standard.fit()

# Сохраним оценку
ATE_dml_standard = dml_standard.coef[0]

In [ ]:
# Данные в формате, необходимом для применения DML
dml_iv_data = dml.DoubleMLData(data   = cps,
                               y_col  = 'lwage',
                               d_cols = 'university',
                               z_cols = 'suniversity',
                               x_cols = ['age', 'health', 'nchild'])

# Метод оценивания E(Z | X)
g_Z = GradientBoostingClassifier(loss          = 'log_loss',
                                 n_estimators  = 100,
                                 learning_rate = 0.1)

# Подготовка объекта
dml_iv = dml.DoubleMLIIVM(obj_dml_data = dml_iv_data,
                          ml_g         = g_Y,
                          ml_m         = g_Z,
                          ml_r         = g_T,
                          n_rep        = 1,
                          n_folds      = 5)

# Оценим параметры
dml_iv.fit()

# Посмотрим на результат
print(dml_iv)

# Сохраним оценку
LATE_dml_iv = dml_iv.coef[0]

In [ ]:
# Альетнавный инструмент как доля людей в штате с высшим образованием в
# соответствующей возрастной группе
cps['stuniversity'] = stats.ave(cps['university'], cps['state'], cps['age'],
                                FUN = base.mean)

# Переведем данную переменную в бинарную
cps['bstuniversity'] = cps['stuniversity'] >= np.median(cps['stuniversity'])

In [ ]:
# Данные в формате, необходимом для применения DML
dml_iv2_data = dml.DoubleMLData(data   = cps,
                                y_col  = 'lwage',
                                d_cols = 'university',
                                z_cols = 'bstuniversity',
                                x_cols = ['age', 'health', 'nchild'])

# Подготовка объекта
dml_iv2 = dml.DoubleMLIIVM(obj_dml_data = dml_iv2_data,
                           ml_g         = g_Y,
                           ml_m         = g_Z,
                           ml_r         = g_T,
                           n_rep        = 1,
                           n_folds      = 5)

# Оценим параметры
dml_iv2.fit()

# Посмотрим на результат
print(dml_iv2)

# Сохраним оценку
LATE_dml_iv2 = dml_iv2.coef[0]

In [ ]:
# Сопоставим результаты
print(pd.DataFrame(data    = [ATE_naive,        ATE_ls,
                              ATE_dml_standard, LATE_dml_iv,
                              LATE_dml_iv2],
                   index   = ['ATE naive',        'ATE ls',
                              'ATE dml standard', 'LATE dml IV',
                              'LATE dml IV2'],
                   columns = ['Оценка']))